# Phase 1 corrections before Phase 2 — Real-Absher
This notebook addresses the instructor's grading feedback. Existing measurements stay historical and exploratory. New results are produced only by executing the corrected experiment.

Implemented: COCO-caption source grouping, recorded source audit with a confidence bound, class-weighted training plus an unweighted scratch control, validation-only selection, a new DFGAN holdout, full and balanced test reports, condition experiments, checkpoint export and independent-reproduction comparison.

Use a **fresh Kaggle GPU runtime with Internet enabled**. Add the corrections ZIP as a Kaggle dataset and also attach the original prepared DANI data (or its `data.zip`). This notebook does not require a W&B login; new training logs are saved offline.

Runtime compatibility: Torch, TorchVision and TorchAudio are all pinned to matching CUDA 12.8 builds. This prevents Kaggle's preinstalled CUDA 13.0 audio extension from breaking Transformers image imports. If updating an already-running kernel, restart Python once after installation, then rerun setup.


In [ ]:
from pathlib import Path
import sys,subprocess,json,shutil
candidates=[]
if (Path.cwd()/'phase1_corrections.py').is_file(): candidates=[Path.cwd()]
elif Path('/kaggle/input').exists(): candidates=sorted({x.parent for x in Path('/kaggle/input').rglob('phase1_corrections.py')})
assert len(candidates)==1, 'Set PROJECT to the corrections folder if no single code dataset was found.'
PROJECT=candidates[0]
print('Corrections source:',PROJECT)
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')])
sys.path.insert(0,str(PROJECT))
# Use a fresh process to check newly installed binary packages.
subprocess.check_call([sys.executable, "-c", "import torch, torchvision, torchaudio; from transformers import AutoImageProcessor, AutoModelForImageClassification; print('Binary/import check: OK', torch.__version__, torchvision.__version__, torchaudio.__version__)"])


## 1. Make a writable copy of the prepared data
The input must include `data/manifest.csv` and the 1,800 original images. Outputs and the newly acquired holdout go under `/kaggle/working`; input datasets are never modified.

In [ ]:
from zipfile import ZipFile
DATA_ROOT=Path('/kaggle/working/phase1_images')
PREPARED=Path('/kaggle/working/phase1_prepared')
RUN=Path('/kaggle/working/phase1_corrected_run')
CACHE=Path('/kaggle/working/dani_publisher_cache')
DATA_ROOT.mkdir(parents=True,exist_ok=True);PREPARED.mkdir(parents=True,exist_ok=True)
if not (DATA_ROOT/'data/manifest.csv').exists():
    manifests=[x for x in Path('/kaggle/input').rglob('manifest.csv') if x.parent.name=='data']
    if len(manifests)==1: shutil.copytree(manifests[0].parent,DATA_ROOT/'data',dirs_exist_ok=True)
    else:
        archives=list(Path('/kaggle/input').rglob('data.zip'))
        assert len(archives)==1, 'Attach the prepared DANI data; set the archive path explicitly if there are several.'
        with ZipFile(archives[0]) as archive:
            for info in archive.infolist():
                if info.is_dir() or not info.filename.startswith('data/') or '/._' in info.filename: continue
                target=(DATA_ROOT/info.filename).resolve()
                assert target.is_relative_to(DATA_ROOT.resolve()), 'Unsafe archive path'
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(archive.read(info))
assert (DATA_ROOT/'data/manifest.csv').is_file()
EVIDENCE=PROJECT/'docs/phase1_corrections/evidence'
print('Data root:',DATA_ROOT)

## 2. Recheck cleaning and split by the true parent image
The publisher filenames encode `COCO_image_id_caption_id`. Every caption link was checked against the publisher's caption dictionary. Real images and all fakes derived from any caption of that image therefore share one group.

Expected original-data result: unreadable 0; duplicate copies removed 402; retained images 1,398; COCO parent IDs 499. One repeated fake connects two parent families, producing 498 connected source groups. The corrected development split has 415 training groups and 83 validation groups. No historical image is called a new test image.

The canonical workflow loads the committed development manifest after checking its checksum and each image hash, label and source group. It does not regenerate the split with the installed scikit-learn version. The reference was generated under scikit-learn 1.8.0; the pinned 1.6.1 shuffled splitter otherwise yields different assignments. No model scores were used to choose this fixed manifest.


In [ ]:
import pandas as pd
from phase1_corrections import inspect_images,attach_links,development_split,audit_report,fingerprint
clean,counts=inspect_images(DATA_ROOT/'data/manifest.csv',DATA_ROOT,PREPARED/'cleaning')
links=pd.read_csv(EVIDENCE/'dani_source_links.csv')
all_old=attach_links(pd.read_csv(PREPARED/'cleaning/all_readable_manifest.csv',dtype={'phash':str}),links)
all_old.to_csv(PREPARED/'all_historical_linked_manifest.csv',index=False)
import hashlib

# Load the fixed split shipped with the corrections dataset.
split_file = EVIDENCE / "corrected_development_manifest.csv"
assert hashlib.sha256(split_file.read_bytes()).hexdigest() == "15f17c6e5d7f6c5df0ce4ac0a6f09023b797e6710e3dc64870bfb3ecbf0ad138", "Fixed split file changed"
frozen = pd.read_csv(split_file, dtype={"sha256": str, "phash": str})

# Verify the fixed records against the images checked in this run.
assert frozen.relative_path.is_unique and frozen.sha256.is_unique
assert set(frozen.sha256) == set(clean.sha256), "Retained image population changed"
observed = all_old.set_index("relative_path").reindex(frozen.relative_path)
for field in ["sha256", "label", "group_id"]:
    assert observed[field].tolist() == frozen[field].tolist(), f"Image identity mismatch: {field}"

# Save this exact split for training and holdout preparation.
development = frozen.copy()
assert set(development.split) == {"train", "val"}
assert development.groupby("group_id").split.nunique().max() == 1
assert development.group_id.nunique() == 498
development.to_csv(PREPARED / "development_manifest.csv", index=False)

print(counts)
print(development.groupby('split').agg(images=('label','size'),groups=('group_id','nunique')))
print(pd.crosstab(development.split,development.label))
assert development.groupby('group_id').split.nunique().max()==1
assert development.group_id.nunique()==498

## 3. Verify the recorded source audit
The committed CSV contains 100 independent publisher-reference decisions and per-image evidence. It is an automated provenance audit, with no human review claimed. All 100 original publisher images matched the local pixels exactly. Zero disagreements imply a **one-sided 95% Clopper–Pearson upper bound of 2.9513%** for source-label disagreement in the fixed random sample's population.

In [ ]:
fixed=pd.read_csv(EVIDENCE/'label_audit_fixed_sample.csv')
decisions=pd.read_csv(EVIDENCE/'label_audit.csv')
for row in fixed.itertuples():
    assert fingerprint(DATA_ROOT/row.relative_path)[0]==row.sha256, 'Audit image identity changed'
report=audit_report(fixed,decisions)
assert report['complete'], 'Complete the fixed audit before training'
print(report)
decisions.to_csv(PREPARED/'label_audit.csv',index=False)
(PREPARED/'label_audit_report.json').write_text(json.dumps(report,indent=2))

## 4. Acquire the new test cohort before training
Only previously unused COCO parents are eligible. DFGAN is held out from development, whose fake images are GALIP. The final reservation stays within indoor/outdoor scenes. The full cohort is naturally imbalanced; a balanced secondary result is derived from the same final predictions.

The first publisher shard is approximately 523 MB and is cached once. The large real-image shard is read by byte ranges, not downloaded in full. Acquisition rejects old parent groups, old pixels and near-duplicate candidates. Do not inspect model predictions until validation selection is locked.

In [ ]:
from dani_provenance import acquire_holdout
protocol=acquire_holdout(EVIDENCE/'reserved_holdout_sources.csv',PREPARED/'development_manifest.csv',
    PREPARED/'all_historical_linked_manifest.csv',DATA_ROOT,CACHE,PREPARED)
PROTOCOL=PREPARED/'protocol_manifest.csv'
print(protocol.groupby(['split','label']).size())
print(json.loads((PREPARED/'holdout_verification.json').read_text()))
assert protocol.groupby('group_id').split.nunique().max()==1

## 5. Train the controlled comparison and lock the selection
Run scratch unweighted, scratch weighted, frozen weighted and full fine-tuning weighted. Both scratch runs have the same seed, architecture, batch size and data. Weights come from training counts only. Weighted per-example loss uses a fixed denominator, so weighting still has an effect at batch size one.

All checkpoints are compared using **unweighted validation cross-entropy**. Validation macro-F1 and both class recalls are now logged. No test metric participates in selection. Physical batch size is eight for both scratch variants; changes relative to the old batch-one run cannot be attributed solely to weighting.

In [ ]:
import torch
from phase1_training import train_and_lock,DEFAULTS
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before this training cell.'
OPERATOR_NAME='Fahad Falah H. AlQahtani'  # Another member must change this for the independent run.
if (RUN/'selection_locked.json').exists():
    assert json.loads((RUN/'config.json').read_text())==DEFAULTS, 'Locked configuration differs'
    comparison=pd.read_csv(RUN/'validation_comparison.csv')
else:
    comparison=pd.DataFrame(train_and_lock(PROTOCOL,DATA_ROOT,RUN))
env=json.loads((RUN/'environment.json').read_text());env['operator']=OPERATOR_NAME
(RUN/'environment.json').write_text(json.dumps(env,indent=2))
print(comparison[['mode','epoch','val_loss','val_accuracy','val_macro_f1','val_real_recall','val_fake_recall']])
print(json.loads((RUN/'selection_locked.json').read_text()))

## 6. Test the proposed condition hypotheses on validation
The historical error analysis did not establish blur, brightness or haze as causes. This cell measures sensitivity to defined interventions: Gaussian blur radius 1.5, brightness factors 0.7/1.3, and a 20% white overlay (synthetic haze). It reports changes from the unmodified validation images, without using the test set to choose a new model.

In [ ]:
from phase1_training import validation_robustness
robustness=validation_robustness(PROTOCOL,DATA_ROOT,RUN)
print(robustness.to_string(index=False))

## 7. Open the final test once and report both populations
Proceed only with the frozen model-selection rule and configuration. Re-executing this cell returns the saved predictions/metrics; it does not choose a model again. The full unbalanced cohort is primary. Balanced secondary results come from exactly the same prediction vectors. Accuracy confidence intervals resample whole COCO parent groups.

After seeing these results, do not tune on this cohort or call it an untouched test for a later experiment. An independent reproduction repeats this fixed experiment; it is not another model-development cycle.

In [ ]:
from phase1_training import final_test
from phase1_corrections import analyse_historical
final=final_test(PROTOCOL,DATA_ROOT,RUN)
print(final.to_string(index=False))
analysis,slices,tests=analyse_historical(RUN,DATA_ROOT,RUN/'error_analysis')
print(slices.loc[slices.dimension.isin(['scene','source_model'])].to_string(index=False))
print('Adjusted condition tests:')
print(tests[['mode','condition','label','p_two_sided','p_holm']].to_string(index=False))

## 8. Export actual weights and results for the grader and Phase 2
Keep the complete Kaggle output directory and the generated ZIP. This includes real `best.pt` files, local model/processor configurations, fixed manifests, selection lock, metrics, logs and environment information. The validation-selected checkpoint can support the Phase 2 classifier; do not assume that the old full model remains the winner.

In [ ]:
BUNDLE=Path('/kaggle/working/Phase1_Corrected_Models.zip')
subprocess.check_call([sys.executable,str(PROJECT/'scripts/export_bundle.py'),'--run',str(RUN),'--output',str(BUNDLE)])
print('Download this actual model bundle:',BUNDLE)

## 9. Independent reproduction — another member
1. Another member opens a new Kaggle runtime, uses this same code, data and configuration, changes `OPERATOR_NAME`, and executes the notebook from top to bottom.
2. Preserve both actual output bundles. Neither a copied CSV nor a rerun in the author's current kernel is an independent member reproduction.
3. Extract both bundles into separate folders and run (use the exact name recorded as the rerun operator):

```bash
python scripts/compare_reproduction.py --original original_run --rerun teammate_run --member "Anoud Mohammd A. Alenzei" --independent-run-confirmed --output reproduction_record
```

Report the original and rerun accuracy, macro-F1 and ECE side by side, including differences. Commit the comparison and the member's run record. The Phase 2 readiness record can be closed only after the real GPU outputs and this independent run exist.